# Clase 15 · Repaso: PennyLane + Qiskit (simulador y hardware real)

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matibilkis/diplomatura-QML-UAI-2026/blob/master/clase_15/0_repaso/0_resumen_clase_3_pennylane_y_qiskit.ipynb)

Repaso de la clase 3 en un solo recorrido:

1. Un circuito de Bell en un **simulador** y en una **computadora cuántica real de IBM** (vía `qiskit.remote`).
2. **Templates**: el *hardware efficient ansatz* genérico (`StronglyEntanglingLayers`).
3. **Optimizadores**: uno estándar de PennyLane y uno custom escrito a mano.
4. **VQE** sobre un Hamiltoniano de Ising, primero simulado y después (opcional) en hardware real.

**Sobre el hardware real:** las secciones que usan IBM están detrás de la bandera `USAR_HARDWARE`. Con `False` (default) todo corre en el simulador y no necesitás cuenta. Con `True` necesitás tu propia API key de IBM Quantum; nunca la escribas en el notebook.

In [ ]:
!pip install -q pennylane pennylane-qiskit qiskit-ibm-runtime tqdm

In [ ]:
import json, os, pathlib
import pennylane as qml
from pennylane import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"PennyLane version: {qml.__version__}")

# Ponelo en True solo si tenés una API key de IBM Quantum
USAR_HARDWARE = False

## 1. Conexión con IBM Quantum (opcional)

`cargar_token` busca la API key, en este orden, en:

1. Secrets de Colab (ícono de llave a la izquierda, nombre `IBM_QUANTUM_TOKEN`),
2. la variable de entorno `IBM_QUANTUM_TOKEN`,
3. un archivo `IBM_QUANTUM_TOKEN` o `apikey*.json` en el directorio actual,
4. un prompt interactivo (`getpass`), que no deja la clave escrita en el notebook.

In [ ]:
def cargar_token():
    try:
        from google.colab import userdata
        return userdata.get("IBM_QUANTUM_TOKEN")
    except Exception:
        pass
    if "IBM_QUANTUM_TOKEN" in os.environ:
        return os.environ["IBM_QUANTUM_TOKEN"]
    archivo_token = pathlib.Path.cwd() / "IBM_QUANTUM_TOKEN"
    if archivo_token.exists():
        return archivo_token.read_text().strip()
    for p in pathlib.Path.cwd().glob("apikey*.json"):
        return json.loads(p.read_text())["apikey"]
    import getpass
    return getpass.getpass("IBM Quantum API key: ")


backend = None
if USAR_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    service = QiskitRuntimeService(channel="ibm_quantum_platform", token=cargar_token())
    backend = service.backend("ibm_marrakesh")
    print(f"backend:   {backend.name}")
    print(f"qubits:    {backend.num_qubits}")
    print(f"operativo: {backend.status().operational}, en cola: {backend.status().pending_jobs}")

## 2. El mismo circuito en simulador y en hardware

Estado de Bell: `H` sobre el qubit 0 y `CNOT(0, 1)`. Lo esperado es 50% en `00` y 50% en `11`.

- `default.qubit` calcula las probabilidades **exactas**.
- `qiskit.remote` manda el circuito a la máquina real. Ahí hay que fijar un número de **shots** (`qml.set_shots`), y el resultado es una estimación con ruido: vas a ver algo de probabilidad en `01` y `10`.

In [ ]:
dev_sim = qml.device("default.qubit", wires=2)


@qml.qnode(dev_sim)
def circuito_sim():
    qml.Hadamard(wires=0)
    qml.CNOT(wires=[0, 1])
    return qml.probs(wires=[0, 1])


print("probabilidades en el simulador (exacto):", circuito_sim())

In [ ]:
if USAR_HARDWARE:
    dev_real = qml.device("qiskit.remote", wires=backend.num_qubits, backend=backend)

    @qml.set_shots(1000)
    @qml.qnode(dev_real)
    def circuito_real():
        qml.Hadamard(wires=0)
        qml.CNOT(wires=[0, 1])
        return qml.probs(wires=[0, 1])

    print("probabilidades en hardware real (1000 shots):", circuito_real())

## 3. Templates: Hardware Efficient Ansatz

`StronglyEntanglingLayers` es un ansatz genérico: cada capa aplica una rotación arbitraria (3 parámetros) a cada qubit y luego una cascada de CNOTs. No sabe nada del problema; es "eficiente" porque usa solo compuertas nativas y poca profundidad.

Los pesos tienen forma `(n_layers, n_wires, 3)`. Para un ansatz basado en química existen templates específicos, por ejemplo [`AllSinglesDoubles`](https://docs.pennylane.ai/en/stable/code/api/pennylane.AllSinglesDoubles.html). Doc del template usado: [`StronglyEntanglingLayers`](https://docs.pennylane.ai/en/stable/code/api/pennylane.StronglyEntanglingLayers.html).

In [ ]:
n_wires = 4
dev = qml.device("default.qubit", wires=n_wires)


@qml.qnode(dev)
def circuito_hea(weights):
    qml.StronglyEntanglingLayers(weights, wires=range(n_wires))
    return qml.state()


forma_hea = qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=n_wires)
weights_hea = np.random.uniform(0, 2 * np.pi, size=forma_hea, requires_grad=True)

print("forma de los pesos (n_layers, n_wires, 3):", forma_hea)
print(qml.draw(circuito_hea)(weights_hea))

## 4. Optimizadores

PennyLane trae optimizadores listos (`AdamOptimizer`, `GradientDescentOptimizer`, ...). Pero un optimizador es solo un objeto con un método `step(cost_fn, params)` que devuelve los parámetros nuevos. Escribir uno propio son tres líneas: descenso por gradiente simple, $\theta \leftarrow \theta - \eta \nabla C(\theta)$.

In [ ]:
# Un optimizador estándar de PennyLane
opt_standard = qml.AdamOptimizer(stepsize=0.1)


# Un optimizador custom
class GDCustom:
    def __init__(self, stepsize=0.1):
        self.stepsize = stepsize

    def step(self, cost_fn, params):
        grad = qml.grad(cost_fn)(params)
        return params - self.stepsize * grad


opt = GDCustom(stepsize=0.2)

## 5. VQE sobre un Hamiltoniano de Ising

Cadena 1D de 4 qubits con borde abierto:

$$H = -J \sum_{i} Z_i Z_{i+1} \; - \; h \sum_i X_i, \qquad J=1,\; h=0.5$$

El costo es $\langle \psi(\theta) | H | \psi(\theta) \rangle$ con $|\psi(\theta)\rangle$ preparado por el HEA. Minimizarlo aproxima la energía del estado fundamental.

In [ ]:
n_wires = 4
J, h = 1.0, 0.5

H_ising = -J * (qml.Z(0) @ qml.Z(1) + qml.Z(1) @ qml.Z(2) + qml.Z(2) @ qml.Z(3)) \
          - h * (qml.X(0) + qml.X(1) + qml.X(2) + qml.X(3))

print("Hamiltoniano de Ising:\n", H_ising)

In [ ]:
dev = qml.device("default.qubit", wires=4)


@qml.qnode(dev)
def costo(weights):
    qml.StronglyEntanglingLayers(weights, wires=range(n_wires))
    return qml.expval(H_ising)


HEA_ansatz = qml.StronglyEntanglingLayers.shape(n_layers=2, n_wires=n_wires)
weights = np.random.uniform(0, 2 * np.pi, size=HEA_ansatz, requires_grad=True)

hist = [float(costo(weights))]
for _ in tqdm(range(100)):
    weights = opt.step(costo, weights)
    hist.append(float(costo(weights)))

print(f"costo inicial: {hist[0]:.4f}")
print(f"costo final:   {hist[-1]:.4f}")

ax = plt.subplot(111)
ax.set_title(r"VQE sobre $H=-\sum_i Z_i Z_{i+1} - 0.5\sum_i X_i$")
ax.set_xlabel("paso")
ax.set_ylabel(r"$\langle H \rangle$")
ax.plot(hist)
plt.show()

Comparación con la energía exacta (diagonalización), para saber qué tan cerca quedó el VQE:

In [ ]:
E0 = np.linalg.eigvalsh(qml.matrix(H_ising, wire_order=range(n_wires)))[0]
print(f"energía exacta del fundamental: {E0:.4f}")
print(f"VQE:                            {hist[-1]:.4f}")

## 6. ¿Y si lo corremos en hardware real?

Mismo costo, pero sobre `qiskit.remote` con 1000 shots y `diff_method="parameter-shift"`.

- El gradiente ya no se puede calcular por retropropagación sobre el estado; se usa la regla de **parameter-shift**: cada derivada parcial cuesta **dos ejecuciones** del circuito (con el parámetro desplazado $\pm\pi/2$).
- Con 24 parámetros (2 capas × 4 qubits × 3) y un Hamiltoniano de varios términos, **cada paso son decenas de jobs** en la cola de IBM. El loop completo de 100 pasos puede tardar horas y consumir buena parte del tiempo gratuito. Por eso acá se deja en pocos pasos; subí `n_pasos_real` bajo tu responsabilidad.

In [ ]:
if USAR_HARDWARE:
    @qml.set_shots(1000)
    @qml.qnode(dev_real, diff_method="parameter-shift")
    def cost_fn_real(weights):
        qml.StronglyEntanglingLayers(weights, wires=range(n_wires))
        return qml.expval(H_ising)

    print("costo en hardware real, con los pesos ya entrenados:", cost_fn_real(weights))

In [ ]:
if USAR_HARDWARE:
    n_pasos_real = 5   # el original usaba 100
    opt_real = GDCustom(stepsize=0.2)
    weights_real = np.random.uniform(0, 2 * np.pi, size=HEA_ansatz, requires_grad=True)

    hist_real = [float(cost_fn_real(weights_real))]
    for i in tqdm(range(n_pasos_real)):
        weights_real = opt_real.step(cost_fn_real, weights_real)
        hist_real.append(float(cost_fn_real(weights_real)))

    print(f"costo inicial (hardware real): {hist_real[0]:.4f}")
    print(f"costo final (hardware real):   {hist_real[-1]:.4f}")

    ax = plt.subplot(111)
    ax.set_title("VQE en hardware real (ibm_marrakesh), parameter-shift")
    ax.set_xlabel("paso")
    ax.plot(hist_real)
    plt.show()